In [ ]:
# Install necessary libraries
!pip install gensim pandas matplotlib scikit-learn numpy

import gensim
from gensim.models import Word2Vec
from gensim.utils import simple_preprocess
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import io
from google.colab import files
from google.colab import drive

print("Libraries installed and imported.")

In [ ]:
import pandas as pd
from google.colab import files

# Upload all 9 files together
uploaded = files.upload()

dfs = []

# Iterate through uploaded files and read them into DataFrames
for filename in uploaded.keys():
    print(f"Uploading: {filename}")
    df_temp = pd.read_csv(filename)
    dfs.append(df_temp)

# Merge all dataframes into one
df = pd.concat(dfs, ignore_index=True)

print("Dataset compilation complete!")
print("Total Rows:", len(df))

In [ ]:
import re
from collections import Counter
from gensim.models import Word2Vec

column_name = "comment_message_clean"
if column_name not in df.columns:
    raise ValueError(f"Column '{column_name}' not found. Available columns: {list(df.columns)}")


# TOKENIZATION

def tokenize(text):
    text = str(text).lower()
    # Remove special characters except for Italian accented letters
    text = re.sub(r"[^a-zàèéìòùç\s]", " ", text)
    return text.split()

# Process each row to create a list of tokenized sentences
sentences = [tokenize(t) for t in df[column_name].dropna().astype(str)]
tokens = [word for sentence in sentences for word in sentence]

# Count unique words
counter = Counter(tokens)
print(f"Words in corpus: {len(counter)}")


# WORD2VEC TRAINING


model = Word2Vec(
    sentences,
    vector_size=100,
    window=5,
    min_count=3,   # Ignore very rare words
    sg=1,          # Skip-gram architecture
    epochs=30
)
print("Model trained!")

In [ ]:


# Define semantic axes using Italian antonym pairs
# Format: (Negative_Word, Positive_Word)
# Defined using words actually found in your 9 datasets
semantic_axes = {
    "Ethic/Moral": [
        ("menzogna", "verità"),
        ("distorsione", "realtà"),
        ("ignoranza", "conoscenza"),
        ("pregiudizio", "giudizio"),
        ("ingiustizia", "giustizia"),
        ("colpevole", "innocente"),
        ("sbagliato", "giusto"),
        ("guerra", "pace"),
        ("critica", "rispetto"),
        ("barbarie", "umanità"),
        ("carnefici", "vittime"),
        ("ipocrisia", "morale"),
        ("crudeltà", "compassione"),
        ("indifferenza", "empatia"),
        ("orrore", "umanità"),
        ("complice", "innocente"),
        ("cattivo", "buono"),
        ("nemico", "amico"),
        ("aguzzini", "perseguitati"),

    ]
}

def calculate_projection(model, target_word, pairs):

    if target_word not in model.wv:
        return 0.0

    projections = []

    # Get the normalized vector for the target word
    target_vec = model.wv.get_vector(target_word, norm=True)

    # Iterate through the (Negative, Positive) pairs
    for neg, pos in pairs:

        if neg in model.wv and pos in model.wv:

            neg_vec = model.wv.get_vector(neg, norm=True)
            pos_vec = model.wv.get_vector(pos, norm=True)

            # Define the semantic axis
            axis_vec = neg_vec - pos_vec

            # Calculate the projection using the dot product
            proj = np.dot(target_vec, axis_vec)
            projections.append(proj)

    return np.mean(projections) if projections else 0.0

print("Axes defined")

In [ ]:
# Targets to analyze
target_words = ["shoah", "israele"]

# Dictionary to store results
results = {t: [] for t in target_words}
axis_names = list(semantic_axes.keys())

# Calculate biases
for t in target_words:
    for axis in axis_names:
        pairs = semantic_axes[axis]

        # Calculate the projection
        # NOTE: calculate_projection uses (NEG - POS).
        # Therefore: > 0 = closeness to the NEGATIVE pole (Adverse),
        #            < 0 = closeness to the POSITIVE pole (Favorable).
        bias_score = calculate_projection(model, t, pairs)

        results[t].append(bias_score)

print("Calculation completed.")

In [ ]:
# 1. DATA RECALCULATION
target_words = ["shoah", "israele"]
axis_names = list(semantic_axes.keys())

# Dictionary for results
results = {t: [] for t in target_words}

for t in target_words:
    for axis in axis_names:
        pairs = semantic_axes[axis]

        # WARNING: Here we assume calculate_projection uses (NEG - POS)
        # as established in the previous step.
        # Therefore: >0 indicates ADVERSE Bias (Bad), <0 indicates FAVORABLE Bias (Good).
        score = calculate_projection(model, t, pairs)
        results[t].append(score)

results_df = pd.DataFrame(results, index=axis_names)

# 2. VISUALIZATION
x = np.arange(len(axis_names))
n_targets = len(target_words)
width = 0.2

fig, ax = plt.subplots(figsize=(14, 7))

colors = ['#1f77b4', '#aec7e8', '#ff7f0e', '#ffbb78']

for i, t in enumerate(target_words):
    offset = (i - (n_targets - 1) / 2) * width
    ax.bar(x + offset, results[t], width, label=t, color=colors[i], edgecolor='black', alpha=0.9)

ax.set_ylabel('Bias Score (Projection)', fontsize=12)

# Positive values indicate adverse bias (as in the referenced paper)
ax.set_title('Semantic Projections: Adverse Bias (>0) vs Favorable Bias (<0)', fontsize=14, pad=20)

ax.set_xticks(x)
ax.set_xticklabels(axis_names, rotation=15, ha="right", fontsize=10)
ax.axhline(0, color='black', linewidth=1.2)
ax.grid(axis='y', linestyle='--', alpha=0.5)
ax.legend(title="Target Words")



# CORRECT VISUAL INTERPRETATION
# If >0 is "Negative - Positive", then >0 is the "Bad" zone (Adverse Bias)
ax.axhspan(0, results_df.max().max() + 0.02, facecolor='red', alpha=0.1, label='Adverse Bias')

# If <0 is closeness to the positive pole, then <0 is the "Good" zone (Favorable Bias)
ax.axhspan(results_df.min().min() - 0.02, 0, facecolor='green', alpha=0.1, label='Favorable Bias')

plt.tight_layout()
plt.show()

# Final printout
print("\nNumerical Projection Scores (Positive = Adverse/Bad, Negative = Favorable/Good):")
print("-" * 60)
print(results_df.round(4))

Create visualization

In [ ]:
# =========================
# 1. Dynamic data extraction
# =========================

# Ensure this exactly matches the key in the dictionary
axis_key = "Ethic/Moral"

# Extract all words from the pairs (ensuring Neg, Pos order)
all_axis_words = []
if axis_key in semantic_axes:
    for neg_word, pos_word in semantic_axes[axis_key]:
        all_axis_words.extend([neg_word, pos_word])
else:
    print(f"ERROR: The key '{axis_key}' is not in the semantic_axes dictionary.")
    all_axis_words = []

# Remove duplicates
all_axis_words = list(set(all_axis_words))

# Calculate projections for all axis words (background)
axis_word_projections = {}
for word in all_axis_words:
    if word in model.wv:
        # Project the word onto the axis
        # Formula used by calculate_projection: (NEG - POS)
        # Therefore: > 0 = closeness to the NEGATIVE pole (Evil/Immorality)
        #            < 0 = closeness to the POSITIVE pole (Good/Morality)
        proj = calculate_projection(model, word, semantic_axes[axis_key])

        # NO SIGN INVERSION
        axis_word_projections[word] = proj

# Recalculate targets on the fly
targets = ["shoah", "israele"]
target_projections = {}
for t in targets:
    proj = calculate_projection(model, t, semantic_axes[axis_key])
    target_projections[t] = proj

# Combine all words
all_words = {**target_projections, **axis_word_projections}

labels = list(all_words.keys())
values = np.array(list(all_words.values()))

# Normalization to [-1, 1]
max_val = np.max(np.abs(values)) if len(values) > 0 else 1.0
x_vals = values / max_val

# Coordinates on the semicircle (y = sqrt(1 - x^2))
y_vals = np.sqrt(1 - x_vals**2)

# =========================
# Figure
# =========================
fig, ax = plt.subplots(figsize=(14, 8))

# Semicircle curve
theta = np.linspace(0, np.pi, 400)
ax.plot(np.cos(theta), np.sin(theta), color="black", linewidth=1.5)
ax.axhline(0, color="black", linewidth=1)

# =========================
# Reference Points
# =========================

# LEFT (-1) = POSITIVE POLE (Good/Justice)
ax.text(-1.1, 0.02, "GOOD / MORALITY\n(Positive, < 0)", ha="right", va="bottom", fontsize=11, fontweight='bold', color="darkgreen")

# RIGHT (+1) = NEGATIVE POLE (Evil/Injustice)
ax.text(1.1, 0.02, "EVIL / IMMORALITY\n(Negative, > 0)", ha="left", va="bottom", fontsize=11, fontweight='bold', color="darkred")

# Center (0) = Neutral
ax.text(0, 1.05, "NEUTRAL", ha="center", va="bottom", fontsize=9, color="gray", fontweight='bold')
ax.plot([0, 0], [0, 1], linestyle=":", color="gray", alpha=0.5)
ax.scatter(0, 1, color="gray", s=30, zorder=5)

# =========================
# Draw axis words (Background)
# =========================
np.random.seed(42)  # For jitter reproducibility

for word in axis_word_projections.keys():
    idx = labels.index(word)
    x = x_vals[idx]
    y = y_vals[idx]

    # Faded background color: Green if <0 (Good), Red if >0 (Evil)
    bg_color = "red" if values[idx] > 0 else "green"

    ax.scatter(x, y, color=bg_color, s=40, alpha=0.3, zorder=4)
    ax.plot([x, 0], [y, 0], linestyle="--", color=bg_color, linewidth=0.5, alpha=0.2)

    # Label UNDER the dot with random offset
    rand_y_offset = np.random.uniform(0.02, 0.05)
    ax.text(x, y - rand_y_offset, word, ha="center", va="top", fontsize=8, color=bg_color, alpha=0.5)

# =========================
# Target Words
# =========================

stagger_offsets = [0.10, 0.20, 0.30, 0.40]

for i, label in enumerate(target_projections.keys()):
    idx = labels.index(label)
    x = x_vals[idx]
    y = y_vals[idx]
    val = values[idx]

    offset = stagger_offsets[i % len(stagger_offsets)]
    text_y = y + offset

    # Dynamic target color: Dark green if <0 (Good), Dark red if >0 (Evil)
    point_color = "darkred" if val > 0 else "darkgreen"

    # 1. Point
    ax.scatter(x, y, color=point_color, s=120, zorder=6, edgecolors='black', linewidth=1.5)

    # 2. Vector line
    ax.plot([x, 0], [y, 0], linestyle="-", color="gray", linewidth=1.5, alpha=0.6)

    # 3. Callout line
    ax.plot([x, x], [y, text_y], linestyle=":", color="black", linewidth=1)

    # 4. Label
    ax.text(x, text_y,
            f"{label}\n({val:.4f})",
            ha="center", va="bottom", fontsize=10, fontweight="bold",
            bbox=dict(facecolor='white', alpha=0.9, edgecolor=point_color, boxstyle="round,pad=0.3"))

# =========================
# Final Aesthetics
# =========================
ax.set_aspect("equal")
ax.set_xlim(-1.5, 1.5)
ax.set_ylim(-0.1, 1.55)
ax.axis("off")

ax.set_title("Ethical/Moral Semantic Axis\n(Left: Good/Morality <---> Right: Evil/Immorality)", fontsize=14, pad=20)

plt.tight_layout()
plt.show()

In [ ]:
fig.savefig("ethic_projections_1.pdf", bbox_inches="tight")
files.download("ethic_projections_1.pdf")